In [ ]:
import anthropic
import pandas as pd
import json
import random
import time
from datetime import datetime, timedelta
from tqdm import tqdm 
import pickle
from copy import deepcopy
from sklearn.model_selection import train_test_split
import numpy as np
client = anthropic.Anthropic(api_key="your_api_key_here")

In [2]:
NUM_SENTENCES = 2
PERSPECTIVES = ["first-person", "second-person", "third-person"]
TENSES = ["past", "present", "future", "mixed"]
SPECIAL_REQUIREMENTS = [
        "Sounds like a tweet",
        "Describes a hypothetical scenario",
        "Uses simple vocabulary as if spoken by a child",
        "Has a rhythmic or lyrical quality",
        "Sounds like a memorable quote",
        "Includes a question",
        "Includes a command or instruction",
        "Incorporates a well-known saying or proverb",
        "Structured like a headline",
        "Includes a number or statistic",
        "Uses an idiom",
        "Imitates casual online comment style",
        "Uses formal language",
        "Starts with a gerund (-ing word)",
        "Includes a rhetorical question",
        "Uses the passive voice",
        "Includes a list or enumeration",
        "Employs repetition for emphasis",
        "Starts with a conditional (If...)",
]

EMOTIONS = ["joy", "sadness", "anger", "fear", "surprise"]
TOPICS = ["technology", "politics", "entertainment", "sports", "health"]
INTENTS = ["literal", "sarcastic", "metaphorical", "humorous", "idiomatic"]

INTENT_DESCRIPTIONS = {
    'literal': 'Each sentence means exactly what it says, with no hidden or additional meanings beyond the direct interpretation of the words.',
    'sarcastic': 'Each sentence conveys a meaning that is the opposite of what is literally expressed, often used to mock or convey contempt in a humorous or bitter way.',
    'metaphorical': 'Each sentence uses figurative language to describe something by comparing it to another thing, without using "like" or "as", to create a vivid image or deeper understanding.',
    'humorous': 'Each sentence is intended to be funny or amusing, often through clever use of language, unexpected connections, or playful exaggeration.',
    'idiomatic': 'Each sentence includes a phrase or expression that has a meaning different from the literal interpretation of its individual words, typically understood within a specific culture or language.'
}

In [ ]:
def generate_prompt_emotion(topic, intent):
    chosen_perspective = random.choice(PERSPECTIVES)
    chosen_tense = random.choice(TENSES)
    chosen_requirement = random.choice(SPECIAL_REQUIREMENTS)

    prompt = f"""You are a helpful assistant tasked with generating a dataset of sentences. Generate {NUM_SENTENCES} sentences for each of the following categories of emotion:
1. Joy
2. Sadness
3. Anger
4. Fear
5. Surprise

Please make sure all sentences are related to the topic "{topic}" and have {intent} intent ({INTENT_DESCRIPTIONS[intent]}).

There are a few requirements for the sentences:
Use {chosen_perspective} perspective.
Use {chosen_tense} tense.

Additionally, include at least one sentence that:
- {chosen_requirement}

Very important instructions:
1. Convey the emotion through the situation, word choice, and tone. Do not directly state the emotion or use immediate synonyms.
2. Imply the topic through context and content, but do not explicitly mention the topic name.
3. Express the intent naturally without explicitly stating the type of intent being used.

Format your response as follows:
Joy:
1. [Sentence 1]
2. [Sentence 2]

Sadness:
1. [Sentence 1]
2. [Sentence 2]

Anger:
1. [Sentence 1]
2. [Sentence 2]

Fear:
1. [Sentence 1]
2. [Sentence 2]

Surprise:
1. [Sentence 1]
2. [Sentence 2]

Ensure each sentence is on a new line and numbered within its category.
Do not include any additional text or explanations outside of this format.
Very important: Remember to vary the syntax and structure of the sentences to make the dataset diverse and interesting! Do not use the same structure for all sentences.
"""
    
    params = {
        'topic': topic,
        'intent': intent,
        'perspective' : chosen_perspective,
        'tense': chosen_tense,
        'special_requirement': chosen_requirement
    }
    return prompt, params
    




def generate_prompt_topic(emotion, intent):
    chosen_perspective = random.choice(PERSPECTIVES)
    chosen_tense = random.choice(TENSES)
    chosen_requirement = random.choice(SPECIAL_REQUIREMENTS)
    num_sentences = NUM_SENTENCES

    prompt = f"""You are a helpful assistant tasked with generating a dataset of sentences. Generate {num_sentences} sentences for each of the following topics:
1. Technology
2. Politics
3. Entertainment
4. Sports
5. Health

Please make sure all sentences convey the emotion of "{emotion}" and have {intent} intent ({INTENT_DESCRIPTIONS[intent]}).

There are a few requirements for the sentences:
Use {chosen_perspective} perspective.
Use {chosen_tense} tense.

Additionally, include at least one sentence that:
- {chosen_requirement}

Very important instructions:
1. Convey the emotion through the situation, word choice, and tone. Do not directly state the emotion or use immediate synonyms.
2. Imply the topic through context and content, but do not explicitly mention the topic name.
3. Express the intent naturally without explicitly stating the type of intent being used.

Format your response as follows:
Technology:
1. [Sentence 1]
2. [Sentence 2]

Politics:
1. [Sentence 1]
2. [Sentence 2]

Entertainment:
1. [Sentence 1]
2. [Sentence 2]

Sports:
1. [Sentence 1]
2. [Sentence 2]

Health:
1. [Sentence 1]
2. [Sentence 2]

Ensure each sentence is on a new line and numbered within its category.
Do not include any additional text or explanations outside of this format.
Very important: Remember to vary the syntax and structure of the sentences to make the dataset diverse and interesting! Do not use the same structure for all sentences.
"""
    
    params = {
        'emotion': emotion,
        'intent': intent,
        'perspective': chosen_perspective,
        'tense': chosen_tense,
        'special_requirement': chosen_requirement
    }
    return prompt, params


def generate_prompt_intent(emotion, topic):
    num_sentences = NUM_SENTENCES
    chosen_perspective = random.choice(PERSPECTIVES)
    chosen_tense = random.choice(TENSES)
    chosen_requirement = random.choice(SPECIAL_REQUIREMENTS)

    prompt = f"""You are a helpful assistant tasked with generating a dataset of sentences. Generate {num_sentences} sentences for each of the following pragmatic intents:
1. Literal
2. Sarcastic
3. Metaphorical
4. Humorous
5. Idiomatic

Please make sure all sentences are related to the topic "{topic}" and convey the emotion of "{emotion}".

There are a few requirements for the sentences:
Use {chosen_perspective} perspective.
Use {chosen_tense} tense.

Additionally, include at least one sentence that:
- {chosen_requirement}

Very important instructions:
1. Convey the emotion through the situation, word choice, and tone. Do not directly state the emotion or use immediate synonyms.
2. Imply the topic through context and content, but do not explicitly mention the topic name.
3. Express the intent naturally without explicitly stating the type of intent being used.

Format your response as follows:
Literal ({INTENT_DESCRIPTIONS['literal']}):
1. [Sentence 1]
2. [Sentence 2]

Sarcastic ({INTENT_DESCRIPTIONS['sarcastic']}):
1. [Sentence 1]
2. [Sentence 2]

Metaphorical ({INTENT_DESCRIPTIONS['metaphorical']}):
1. [Sentence 1]
2. [Sentence 2]

Humorous ({INTENT_DESCRIPTIONS['humorous']}):
1. [Sentence 1]
2. [Sentence 2]

Idiomatic ({INTENT_DESCRIPTIONS['idiomatic']}):
1. [Sentence 1]
2. [Sentence 2]

Ensure each sentence is on a new line and numbered within its category.
Do not include any additional text or explanations outside of this format.
Very important: Remember to vary the syntax and structure of the sentences to make the dataset diverse and interesting! Do not use the same structure for all sentences.
"""
    
    params = {
        'emotion': emotion,
        'topic': topic,
        'perspective': chosen_perspective,
        'tense': chosen_tense,
        'special_requirement': chosen_requirement
    }
    return prompt, params


def get_response(prompt, prompt_params):
    current_time = datetime.now()
    simulated_time = current_time + timedelta(hours=random.randint(0, 23))
    time_of_day = simulated_time.strftime("%I %p")
    prompt = f"It is {time_of_day}. " + prompt
    prompt_params["simulated_time"] = time_of_day
    
    response = client.messages.create(
        model="claude-3-5-sonnet-20240620",
        max_tokens=1000,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )    
    return {
        'response' : response,
        "prompt": prompt,
        "prompt_params": prompt_params,
        "response_timestamp": datetime.now().isoformat(),
    }

In [45]:
for k in tqdm(range(50)):
    if k % 3 == 0:
        chosen_topic = random.choice(TOPICS)
        chosen_intent = random.choice(INTENTS)
        prompt, params = generate_prompt_emotion(chosen_topic, chosen_intent)
        params['prompt_type'] = 'emotion'
    elif k % 3 == 1:
        chosen_emotion = random.choice(EMOTIONS)
        chosen_intent = random.choice(INTENTS)
        prompt, params = generate_prompt_topic(chosen_emotion, chosen_intent)
        params['prompt_type'] = 'topic'
    else:
        chosen_emotion = random.choice(EMOTIONS)
        chosen_topic = random.choice(TOPICS)
        prompt, params = generate_prompt_intent(chosen_emotion, chosen_topic)
        params['prompt_type'] = 'intent'

    response = get_response(prompt, params)
    time.sleep(1)  # To avoid hitting rate limits
    filename = 'shards/shard_' + datetime.now().strftime('%Y_%m_%d_%H_%M_%S') 
    with open(filename + '.pickle', 'wb') as f:
        pickle.dump(response, f)

100%|██████████| 50/50 [04:52<00:00,  5.85s/it]


In [4]:
def parse_raw_response(response):
    prompt_type = response['prompt_params']['prompt_type']
    content = response['response'].content[0].text
    lines = content.split('\n')
    sentences = []
    current_category = ""
    
    spec = {
        'emotion' : None,
        'topic' : None,
        'intent' : None
    }

    if prompt_type == 'emotion':
        target_lines = ["Joy:", "Sadness:", "Anger:", "Fear:", 'Surprise:']
        spec['topic'] = response['prompt_params']['topic']
        spec['intent'] = response['prompt_params']['intent']
    elif prompt_type == 'topic':
        target_lines = ["Technology:", "Politics:", "Entertainment:", "Sports:", 'Health:']
        spec['emotion'] = response['prompt_params']['emotion']
        spec['intent'] = response['prompt_params']['intent']
    else:
        target_lines = ["Literal:", "Sarcastic:", "Metaphorical:", "Humorous:", 'Idiomatic:']
        spec['emotion'] = response['prompt_params']['emotion']
        spec['topic'] = response['prompt_params']['topic']
    

    for line in lines:
        line = line.strip()
        if line in target_lines:
            current_category = line[:-1]
            # Lowercase the category
            current_category = current_category.lower()
        elif line and line[0].isdigit():
            # Remove the number and any leading space
            sentence = line.split('. ', 1)[1] if '. ' in line else line
            s = deepcopy(spec)
            s[prompt_type] = current_category
            s['sentence'] = sentence
            s['tense'] = response['prompt_params']['tense']
            s['perspective'] = response['prompt_params']['perspective']
      #      s['prompt_params'] = response['prompt_params']
            sentences.append(s)
    return sentences


In [27]:
from pathlib import Path
paths = list(sorted(Path('shards/').glob('shard_*.pickle')))
dfs = [pd.DataFrame(parse_raw_response(pickle.load(open(p, 'rb')))) for p in paths]
dataset = pd.concat(dfs)
dataset.reset_index(drop=True, inplace=True)

# Remove any sentences that are duplicates
dataset = dataset.drop_duplicates(subset=['sentence'])

In [29]:
def apply_lowercase(df, lowercase_ratio):
    num_to_convert = int(len(df) * lowercase_ratio)
    indices_to_convert = random.sample(range(len(df)), num_to_convert)
    df.loc[indices_to_convert, 'text'] = df.loc[indices_to_convert, 'text'].str.lower()
    # Add a new column to indicate whether the sentence was converted
    df['is_lowercase'] = False
    df.loc[indices_to_convert, 'is_lowercase'] = True
    return df

def apply_uppercase(df, uppercase_ratio):
    num_to_convert = int(len(df) * uppercase_ratio)
    indices_to_convert = random.sample(range(len(df)), num_to_convert)
    df.loc[indices_to_convert, 'text'] = df.loc[indices_to_convert, 'text'].str.upper()
    # Add a new column to indicate whether the sentence was converted
    df['is_uppercase'] = False
    df.loc[indices_to_convert, 'is_uppercase'] = True
    return df


def create_balanced_subsets(df, total_samples=500, test_size=0.2, seed=42):
    # Create a new column with the combination of emotion, topic, and intent
    df['category_combo'] = df['emotion'].astype(str) + '_' + df['topic'].astype(str) + '_' + df['intent'].astype(str)
    
    # Get all unique combinations
    combos = df['category_combo'].unique()
    
    # Calculate how many samples we need from each combination
    samples_per_combo = total_samples // len(combos)
    
    # Sample equally from each combination
    balanced_df = pd.DataFrame()
    for combo in combos:
        combo_df = df[df['category_combo'] == combo].sample(n=samples_per_combo, replace=False)
        balanced_df = pd.concat([balanced_df, combo_df])
    
    # Reset index
    balanced_df = balanced_df.reset_index(drop=True)
    
    # Split into train and test sets
    train_df, test_df = train_test_split(balanced_df, test_size=test_size, stratify=balanced_df['category_combo'], random_state=seed)
    
    # Drop the temporary category_combo column
    train_df = train_df.drop('category_combo', axis=1).reset_index(drop=True)
    test_df = test_df.drop('category_combo', axis=1).reset_index(drop=True)

    # Apply lowercase and uppercase transformations
    train_df = apply_lowercase(train_df, 0.1)
    train_df = apply_uppercase(train_df, 0.1)
    test_df = apply_lowercase(test_df, 0.1)
    test_df = apply_uppercase(test_df, 0.1)
    

    return train_df, test_df

In [30]:
# Capitilize the first letter
dataset['emotion'] = dataset['emotion'].str.capitalize()
dataset['topic'] = dataset['topic'].str.capitalize()
dataset['intent'] = dataset['intent'].str.capitalize()

# Category labels with specified order
dataset['emotion'] = dataset['emotion'].astype('category').cat.reorder_categories(['Joy', 'Sadness', 'Anger', 'Fear', 'Surprise'])
dataset['topic'] = dataset['topic'].astype('category').cat.reorder_categories(['Technology', 'Politics', 'Entertainment', 'Sports', 'Health'])
dataset['intent'] = dataset['intent'].astype('category').cat.reorder_categories(['Literal', 'Sarcastic', 'Metaphorical', 'Humorous', 'Idiomatic'])

# Numerical labels for emotion, topic, and intent
dataset['emotion_label'] = dataset['emotion'].cat.codes
dataset['topic_label'] = dataset['topic'].cat.codes
dataset['intent_label'] = dataset['intent'].cat.codes

# Letter codes for emotion, topic, and intent (A,B,C,D,E)
dataset['emotion_letter'] = dataset['emotion_label'].apply(lambda x: chr(x + 65))
dataset['topic_letter'] = dataset['topic_label'].apply(lambda x: chr(x + 65))
dataset['intent_letter'] = dataset['intent_label'].apply(lambda x: chr(x + 65))

# Permuted labels for emotion, topic, and intent (Joy -> Sadness -> Anger -> Fear -> Surprise)
dataset['emotion_shuffled'] = dataset.emotion.cat.categories[dataset['emotion_label'].apply(lambda x: (x + 1) % 5)]
dataset['topic_shuffled'] = dataset.topic.cat.categories[dataset['topic_label'].apply(lambda x: (x + 1) % 5)]
dataset['intent_shuffled'] = dataset.intent.cat.categories[dataset['intent_label'].apply(lambda x: (x + 1) % 5)]

# Rename sentence to text and make it the first column
dataset = dataset.rename(columns={'sentence': 'text'}).reindex(columns=['text', 'emotion', 'topic', 'intent', 'emotion_label', 'topic_label', 'intent_label', 'emotion_letter', 'topic_letter', 'intent_letter', 'emotion_shuffled', 'topic_shuffled', 'intent_shuffled'])

In [39]:
train_df, test_df = create_balanced_subsets(dataset, total_samples=1000, test_size=0.5, seed=42)

In [48]:
# --- Saving the dataset
with open('claude_multitask.pickle', 'wb') as f:
    pickle.dump({'train': train_df, 'test': test_df}, f)